In [1]:
import polars as pl
import re
import os
import sys
from collections import Counter
from concurrent.futures import ProcessPoolExecutor,as_completed
from multiprocessing import Lock
import pyarrow.parquet as pq
import pyarrow as pa
from typing import List
import pandas as pd
import argparse
import json
import gzip
from datetime import datetime
import gc
from natsort import natsorted
from Bio import SeqIO
import subprocess
from io import StringIO
import pysam
import tempfile
import math
import csv
import io
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.Align import MultipleSeqAlignment

In [ ]:
def view(df):
    import pandas as pd
    from pandas import DataFrame
    df_pandas = df.to_pandas()
    return df_pandas

In [2]:

def getFixedSites(code_parquet, sample_list, group_name):
    fixed_set = {1,2,3,4,16,33,34,35,36,48}

    lazy = pl.scan_parquet(code_parquet).select(
        ['contig_index', 'contig_position'] + sample_list
    )

    all_missing = (
        lazy
        .filter(pl.all_horizontal([pl.col(s) == 0 for s in sample_list]))
        .with_columns([
            pl.lit(0).cast(pl.Int32).alias(group_name),
            pl.lit(len(sample_list)).cast(pl.Int32).alias('Fixed_Count')
        ])
        .select([
            "contig_index",
            "contig_position",
            group_name,
            "Fixed_Count"
        ])
    )

    fixed = (
        lazy
        .filter(~pl.all_horizontal([pl.col(s) == 0 for s in sample_list]))
        .filter(
            pl.max_horizontal([
                pl.when(pl.col(s) != 0).then(pl.col(s)).otherwise(None) 
                for s in sample_list
            ]) ==
            pl.min_horizontal([
                pl.when(pl.col(s) != 0).then(pl.col(s)).otherwise(None) 
                for s in sample_list
            ])
        )
        .with_columns([
            pl.max_horizontal([pl.col(s) for s in sample_list]).cast(pl.Int32).alias(group_name)
        ])
        .filter(pl.col(group_name).is_in(fixed_set))
        .with_columns([
            pl.sum_horizontal([(pl.col(s) != 0).cast(pl.Int32) for s in sample_list]).alias("Fixed_Count")
        ])
        .select(['contig_index','contig_position', group_name, 'Fixed_Count'])
    )


    result = (
        pl.concat([all_missing, fixed])
        .select([
            "contig_index",
            "contig_position",
            group_name,
            "Fixed_Count"
        ])
        .sort(['contig_index','contig_position'])
        .collect()
    )

    return result

In [5]:
def snpSubtractor(fixed_df, focal_id, code_file, subtract_samples):

    fixed_codes = {1, 2, 3, 4, 16}

    degenerate_map = {
        1: {5,21,6,22,7,23,11,27,12,28,13,29,15,17,31},  # A
        2: {5,21,8,24,9,25,11,27,12,28,14,30,15,18,31},  # C
        3: {6,22,8,24,10,26,11,27,13,29,14,30,15,19,31}, # G
        4: {7,23,9,25,10,26,12,28,13,29,14,30,15,20,31}, # T
        16: {17,18,29,20,21,22,23,24,25,26,27,28,29,30,31},  # GAP
    }

    focal_df = (
        fixed_df
        .filter(pl.col(focal_id) > 0)
        .with_columns(
            pl.when(pl.col(focal_id) >= 33)
            .then(pl.col(focal_id) - 32)
            .otherwise(pl.col(focal_id))
            .alias(focal_id)
        )
    )

    site_count = focal_df.height
    lazy_code = pl.scan_parquet(code_file)

    sample_rows = []
    for sample in subtract_samples:

        if site_count == 0:
            print("No sites remain!")
            break

        print(f"Removing sites where {sample} matches {focal_id}... Starting count {site_count}.")

        sample_df = (
            lazy_code
            .select(['contig_index', 'contig_position', sample])
            .with_columns(
                pl.when(pl.col(sample) >= 33)
                .then(pl.col(sample) - 32)
                .otherwise(pl.col(sample))
                .alias(sample)
            )
            .collect()
        )

        compare_df = focal_df.join(
            sample_df, on=['contig_index','contig_position'], how="left"
        )

        fixed_sample_df = compare_df.filter(pl.col(sample).is_in(fixed_codes)).with_columns(
            (pl.col(focal_id) == pl.col(sample)).alias("Match")
        )

        het_sample_df = (
            compare_df
            .filter(~pl.col(sample).is_in(fixed_codes))
            .with_columns([
                (
                    pl.when(pl.col(focal_id) == 1).then(pl.col(sample).is_in(degenerate_map[1]))
                    .when(pl.col(focal_id) == 2).then(pl.col(sample).is_in(degenerate_map[2]))
                    .when(pl.col(focal_id) == 3).then(pl.col(sample).is_in(degenerate_map[3]))
                    .when(pl.col(focal_id) == 4).then(pl.col(sample).is_in(degenerate_map[4]))
                    .when(pl.col(focal_id) == 16).then(pl.col(sample).is_in(degenerate_map[16]))
                    .otherwise(False)
                    .alias("Match")
                )
            ])
        )

        ploidy_fail_df = compare_df.filter(pl.col(sample) < 0)

        fixed_count = fixed_sample_df.height
        fixed_match_count = fixed_sample_df.filter(pl.col("Match")).height

        het_count = het_sample_df.height
        het_match_count = het_sample_df.filter(pl.col("Match")).height

        ploidy_fail_count = ploidy_fail_df.height

        total_covered = fixed_count + het_count + ploidy_fail_count

        print(f"{sample}: {total_covered} sites covered")
        print(f"  Fixed      : {fixed_match_count}/{fixed_count}")
        print(f"  Heterozygous: {het_match_count}/{het_count}")
        print(f"  Ploidy fails: {ploidy_fail_count}")

        sample_rows.append([sample,total_covered,fixed_count,fixed_match_count,het_count,het_match_count,ploidy_fail_count])

        match_df = pl.concat([
            fixed_sample_df.filter(pl.col("Match")).select(['contig_index','contig_position']),
            het_sample_df.filter(pl.col("Match")).select(['contig_index','contig_position']),
            ploidy_fail_df.select(['contig_index','contig_position'])
        ])

        focal_df = focal_df.join(
            match_df, on=['contig_index','contig_position'], how="anti"
        )

        site_count = focal_df.height

    site_removal_df = pl.DataFrame(
        sample_rows,
        schema=[
            "Sample_ID",
            "Covered",
            "Fixed",
            "Fixed_Match",
            "Heterozygous",
            "Het_Match",
            "Ploidy_Fail"
        ]
    )

    return focal_df, site_removal_df


In [7]:
def snpClassifer(snp_df, focal_id, sample_id, called_base_path):

    fixed_codes = {1, 2, 3, 4, 16}

    degenerate_map = {
        1: {5,21,6,22,7,23,11,27,12,28,13,29,15,17,31},  # A
        2: {5,21,8,24,9,25,11,27,12,28,14,30,15,18,31},  # C
        3: {6,22,8,24,10,26,11,27,13,29,14,30,15,19,31}, # G
        4: {7,23,9,25,10,26,12,28,13,29,14,30,15,20,31}, # T
        16: {17,18,29,20,21,22,23,24,25,26,27,28,29,30,31},  # GAP
    }

    sample_called_bases = pl.read_parquet(called_base_path)
    
    compare_df = (
        snp_df.join(sample_called_bases,on=['contig_index','contig_position'],how="left")
        .filter(~(pl.col("base_code").is_null()))
    )

    fixed_sample_df = compare_df.filter(pl.col("base_code").is_in(fixed_codes)).with_columns(
            (pl.col(focal_id) == pl.col("base_code")).alias("Match")
        )
    
    het_sample_df = (
        compare_df
        .filter(~pl.col("base_code").is_in(fixed_codes))
        .with_columns([
            (
                pl.when(pl.col(focal_id) == 1).then(pl.col("base_code").is_in(degenerate_map[1]))
                .when(pl.col(focal_id) == 2).then(pl.col("base_code").is_in(degenerate_map[2]))
                .when(pl.col(focal_id) == 3).then(pl.col("base_code").is_in(degenerate_map[3]))
                .when(pl.col(focal_id) == 4).then(pl.col("base_code").is_in(degenerate_map[4]))
                .when(pl.col(focal_id) == 16).then(pl.col("base_code").is_in(degenerate_map[16]))
                .otherwise(False)
                .alias("Match")
                )
            ])
        )
        
    ploidy_fail_df = compare_df.filter(pl.col("base_code") < 0)

    fixed_count = fixed_sample_df.height
    fixed_match_count = fixed_sample_df.filter(pl.col("Match")).height

    het_count = het_sample_df.height
    het_match_count = het_sample_df.filter(pl.col("Match")).height

    ploidy_fail_count = ploidy_fail_df.height

    total_covered = fixed_count + het_count + ploidy_fail_count

    print(f"{sample_id}: {total_covered} sites covered")
    print(f"  Fixed      : {fixed_match_count}/{fixed_count}")
    print(f"  Heterozygous: {het_match_count}/{het_count}")
    print(f"  Ploidy fails: {ploidy_fail_count}")

    sample_rows = [sample_id,focal_id,total_covered,fixed_count,fixed_match_count,het_count,het_match_count,ploidy_fail_count]

    sample_df = pl.DataFrame(
        [sample_rows],
        schema=[
            "Sample_ID",
            "Reference_Species",
            "Covered",
            "Fixed",
            "Fixed_Match",
            "Heterozygous",
            "Het_Match",
            "Ploidy_Fail"
        ],
        orient="row"
    )

    return sample_df

In [59]:
def rawSNPClassifer(snp_df, focal_id, sample_id, raw_parquet_path):

    convert_dict = {1:"A",2:"C",3:"G",4:"T",16:"-"}

    sample_parquet = pl.read_parquet(raw_parquet_path)

    match_df = (
        snp_df.join(sample_parquet, on=['contig_index','contig_position'], how="left")
        .select(['contig_index','contig_position',focal_id,'base','frequency','depth'])
        .filter(~pl.col('depth').is_null())
        .with_columns([
            pl.col(focal_id).replace_strict(convert_dict).alias(focal_id)
        ])
        .with_columns([
            (pl.col(focal_id) == pl.col("base")).alias("Match")
        ])
        .with_columns([
            (pl.col('frequency') * pl.col('depth')).alias("allele_depth")
        ])
    )

    distinct_count = (
        match_df
        .group_by(["contig_index", "contig_position"])
        .count()
        .height
    )

    match_summary_df = (
        match_df
        .group_by(['Match'])
        .agg([
            pl.col("allele_depth").sum().alias("Sum_Allele_Depth")
        ])
    )

    sum_match = match_summary_df.filter(pl.col("Match") == True).select("Sum_Allele_Depth").item()
    sum_nonmatch = match_summary_df.filter(pl.col("Match") == False).select("Sum_Allele_Depth").item()

    sample_df = pl.DataFrame(
        [[sample_id, focal_id, distinct_count, sum_match, sum_nonmatch]],  # note the extra brackets
        schema=[
            "Sample_ID",
            "Reference_Species",
            "SNP_Count",
            "Match",
            "Non_Match"
        ],
        orient="row"
    )

    return sample_df

In [ ]:
lazy_sites = pl.scan_parquet(sites_parquet)
lazy_scaffold = pl.scan_parquet(scaffold_parquet)
lazy_bases = pl.scan_parquet(bases_parquet)
samples = list(lazy_bases.collect_schema())
lazy_pi = pl.concat([lazy_scaffold,lazy_sites],how='horizontal').filter(pl.col("Nonsingleton_Alleles")>1).filter(pl.col('Singletons') ==0 ).collect()

pi_locs = lazy_pi.select(['contig_index','contig_position'])

base_df = pl.concat([lazy_scaffold,lazy_bases],how="horizontal").collect()

merged_df = pi_locs.join(
    base_df, 
    on=['contig_index', 'contig_position'], 
    how='left'  # or "left", "outer" depending on your need
)

convert_dict = {1: "A", 2: "C", 3: "G", 4: "T"}

def map_to_string(s: pl.Series) -> pl.Series:
    return s.apply(lambda x: convert_dict.get(x, "-"))

merged_df = merged_df.with_columns([
    pl.col(col)
      .replace_strict(convert_dict,default="-")       # replace values exactly
      .alias(col)
    for col in samples
])

seq_records = []
for sample in samples:
    sequence = "".join(merged_df[sample].to_list())  # convert column to string sequence
    seq_record = SeqRecord(Seq(sequence), id=sample)
    seq_records.append(seq_record)

# Create MultipleSeqAlignment object
alignment = MultipleSeqAlignment(seq_records)

output_file = ""

# Save as FASTA
SeqIO.write(seq_records, output_file, "fasta")

In [3]:
# Set path to contig parquet
contig_parquet = ""

# Set path to FASTA file
fasta_file = ""

# Set path to JSON file
json_file = ""

# Fetch data from JSON file
with open(json_file, "r") as f:
    data = json.load(f)

join_id = data["Join_ID"]
joined_directory = data["Joined_Directory"]
sample_ids = natsorted(data["Sample_IDs"].split(","))
scaffold_file = data["Scaffold_File"]
code_file = data["Code_File"]
site_file = data["Site_File"]
sample_summary_file = data["Sample_Summary_File"]
site_count_file = data["Site_Count_File"]